In [3]:
import sys
sys.path.append("../")

from src.loaders.document_loader import EnterpriseDocumentLoader
from src.splitters.chunker import EnterpriseChunker
from src.embeddings.embedding_manager import EmbeddingManager
from vector_store.faiss_store import EnterpriseFAISS
from src.retriever.hybrid_retriever import EnterpriseHybridRetriever

from langchain_classic.retrievers.document_compressors import (
    CrossEncoderReranker
)

from langchain_classic.retrievers import (
    ContextualCompressionRetriever
)

from langchain_community.cross_encoders import HuggingFaceCrossEncoder


In [4]:
loader = EnterpriseDocumentLoader()

documents = loader.load_directory(
    "../datasets/military"
)

chunker = EnterpriseChunker()

chunks = chunker.split_documents(
    documents
)

In [5]:
embedding_model = EmbeddingManager(
    provider="ollama",
    model_name="nomic-embed-text"
)

faiss_store = EnterpriseFAISS(
    embedding_model
)

faiss_store.create(chunks)

vector_retriever = faiss_store.retriever(k=20)

hybrid = EnterpriseHybridRetriever(
    chunks,
    vector_retriever,
    bm25_weight=0.4,
    vector_weight=0.6,
    k=20
)

In [6]:
cross_encoder = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-base"
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1514.47it/s]


In [7]:
reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=5
)

In [8]:
compression_retriever = ContextualCompressionRetriever(

    base_retriever=hybrid.hybrid,

    base_compressor=reranker

)

In [9]:
docs = compression_retriever.invoke(
    "Radar Calibration Procedure"
)

len(docs)

5

In [10]:
for i, doc in enumerate(docs):

    print("="*80)

    print("Rank :", i+1)

    print(doc.metadata)

    print()

    print(doc.page_content[:400])

Rank : 1
{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 13, 'chunk_size': 226}

===
           REMEMBER:  RADAR  DISCIPLINE  SAVES  LIVES.  STAY  VIGILANT.  ========================================================================
===
Rank : 2
{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 11, 'chunk_size': 489}

-  Ghost  Perk  Operatives:  Invisible  to  standard  sweeps  -  Cold-Blooded  Operatives:  No  thermal  signature  return   SECTION  4:  ADVANCED  OPERATOR  TECHNIQUES  

In [11]:
docs = hybrid.search(
    "Radar Calibration Procedure"
)

for doc in docs[:5]:

    print(doc.metadata)

    print(doc.page_content[:200])

    print()

{'source': '..\\datasets\\military\\Weapons.csv', 'row': 13, 'filename': 'Weapons.csv', 'extension': '.csv', 'domain': 'military', 'chunk_id': 27, 'chunk_size': 232}
Weapon_ID: WPN-014
Weapon_Name: P90
Category: SMG
Caliber: 5.7x28mm
Fire_Mode: Full-Auto/Semi
Mag_Size: 50
Effective_Range_Meters: 275
Attachment_Slots: 5
Base_Damage: 28
Recoil_Rating: Low-Medium
Unl

{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 12, 'chunk_size': 207}
in  known  enemy  territory,  then  reposition     to  ambush  counter-UAV  teams  -  Overwatch  Rotation:  Coordinate  with  squad  to  maintain  100%  radar     uptime  through  sequential  UAV  dep

{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'R

In [12]:
docs = compression_retriever.invoke(
    "Radar Calibration Procedure"
)

for doc in docs:

    print(doc.metadata)

    print(doc.page_content[:200])

    print()

{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 13, 'chunk_size': 226}
===
           REMEMBER:  RADAR  DISCIPLINE  SAVES  LIVES.  STAY  VIGILANT.  ==================================================

{'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 11, 'chunk_size': 489}
-  Ghost  Perk  Operatives:  Invisible  to  standard  sweeps  -  Cold-Blooded  Operatives:  No  thermal  signature  return   SECTION  4:  ADVANCED  OPERATOR  TECHNIQUES  ------------------------------

{'producer': 

In [13]:
from src.rerankers.reranker import EnterpriseReranker

reranker = EnterpriseReranker(

    hybrid.hybrid,

    top_n=5

)

docs = reranker.search(
    "Radar Calibration Procedure"
)

print(len(docs))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4344.77it/s]


5


In [15]:
from langchain_community.document_compressors import FlashrankRerank

compressor = FlashrankRerank(top_n=5)

compression_retriever = ContextualCompressionRetriever(
    base_retriever=hybrid.hybrid,
    base_compressor=compressor
)

INFO:flashrank.Ranker:Downloading ms-marco-MultiBERT-L-12...
ms-marco-MultiBERT-L-12.zip: 100%|██████████| 98.7M/98.7M [00:34<00:00, 3.01MiB/s]  


In [16]:
docs = compression_retriever.invoke(
    "Radar Calibration Procedure"
)

for doc in docs:

    print(doc.metadata)

    print(doc.page_content[:200])

    print()

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


{'id': 18, 'relevance_score': np.float32(0.00022855995), 'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 8, 'chunk_size': 269}
SECTION  1:  SYSTEM  OVERVIEW  ---------------------------  The  XR-7  "Sky  Eye"  provides  real-time  aerial  surveillance  with  a  sweep   radius  of  450  meters.  Enemy  positions  are  relayed 

{'id': 17, 'relevance_score': np.float32(0.00020800527), 'producer': 'Skia/PDF m151 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Radar_Manual', 'source': '..\\datasets\\military\\Radar_Manual.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'filename': 'Radar_Manual.pdf', 'extension': '.pdf', 'domain': 'military', 'chunk_id': 10, 'chunk_size': 496}
Killstreak  Active  (Flashing  